# Unsupervised Text Classification

Using LLMs to categorize responses seems to be pretty unreliable and I have been unsuccessful in getting the models to cleanly work. 
In this document, I discuss using `BERTopic` to create categories of responses. This seems to be a fairly popular method in recent years.



## Questions

All responses can be found in `data/student-responses.csv`. In this file, all collected responses are stored from the bar chart and heatmap experiments. In the data cleaning script, a flag for bar chart or heatmap experiment was included.

In [ ]:
# Import modules
import pandas as pd

# Read dataset
df = pd.read_csv("../data/student-responses.csv")

# Filter only Bar Chart experiment responses
df = df[df["experiment"] == "Bar chart"]
df.head()

In [ ]:
df["section"].value_counts().to_frame(name="count").reset_index().rename(columns={"index": "section"}).sort_values("section")

Next, it is worth noting that the quesitons are not in order. Below is a table summary of the questions and their corresponding modules.

| Module | Question Number | Dataframe Column Index (0-index) | Prompt |
|---|---:|---:|---|
| pre-experiment | Q1 | 13 | In this class, you’ll be learning about the process of scientific investigation. What do you think that process looks like, from the perspective of a researcher, compared to what it looks like from the perspective of someone in the general public who is a consumer of scientific results? Write a paragraph (at least 3-5 sentences) about how you think science happens. |
| post-experiment | Q2 | 8 | What do you think the purpose of the experiment was? |
| post-experiment | Q3 | 10 | What hypotheses might the experimenter have been testing? |
| post-experiment | Q4 | 11 | What sources of error are involved in this experiment? |
| post-experiment | Q5 | 12 | What variables were examined? For each variable, identify whether it was quantitative or categorical. |
| post-experiment | Q6 | 9 | What elements of experimental design, such as randomization or the use of a control group, do you think were present in the experiment? Why? |
| abstract reflection | Q7 | 4 | What components of the experiment are clearer now than they were as a participant? What questions do you still have for the experimenter? Write 3-5 sentences reflecting on the abstract. |
| presentation reflection | Q8 | 14 | How did the information you gained from the components of this project (participation, post-study reflection, extended abstract, presentation) differ? |
| presentation reflection | Q9 | 16 | What components were emphasized in the presentation that weren’t emphasized in the abstract? Why do you think that is? |
| presentation reflection | Q10 | 17 | What critiques do you have of this study and its design? What would have made the study better? |
| presentation reflection | Q11 | 15 | If you had to hear about this study using only the extended abstract or only the presentation, which one would you prefer? Which one would be better for determining whether the experiment was well designed? |

## BERTopic

BERTopic is an unsupervised text classification method that leverages clustering and dimensional reduction. 

**Webpage:** <https://maartengr.github.io/BERTopic/index.html>

1. 

**Guide:** <https://www.youtube.com/watch?v=v3SePt3fr9g>

In [ ]:
# Modules for topic modeling
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
import hdbscan
from umap import UMAP
import openai
from bertopic.representation import OpenAI

# UMAP parameters and seed for reproducibility (I think these are the defaults, but I mostly wanted to set the random state)
umap_model = UMAP(n_neighbors=15, 
                  n_components=5, 
                  min_dist=0.0, 
                  metric='cosine', 
                  random_state=42)


# Configure HDBSCAN to have more clusters
hdb = hdbscan.HDBSCAN(
    min_cluster_size=10,     # ↓ smaller = more clusters
    min_samples=2,          # ↓ more sensitive
    prediction_data=True,   # required for some BERTopic visualizations
    cluster_selection_epsilon=0.1  # encourages splitting
)

# Configure OpenAI for topic representation
client = openai.OpenAI(
    base_url="http://localhost:11434/v1",
    api_key='ollama'
)

representation_model = OpenAI(client, model='mistral')

# Fit model for Q1 responses
q1 = df.iloc[:, 15].dropna().astype(str).tolist()
topic_model = BERTopic(
    embedding_model="all-MiniLM-L6-v2", 
    umap_model=umap_model,
    n_gram_range = (1,5), # I doubt there will be n-grams with 5 words, but it may be helpful
    calculate_probabilities=True,
    hdbscan_model=hdb,
    verbose=True,
    representation_model=representation_model
)
topics, probs = topic_model.fit_transform(q1)


In [ ]:
topic_model.get_topic_info()

In [ ]:
# Perform hierarchical topic reduction
hierarchical_topics = topic_model.hierarchical_topics(q1)

# Visualize the subtopics
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

In [ ]:
topic_model.visualize_topics()

In [ ]:
topic_model.visualize_documents(q1)

## Function that fits model for specified question

In [37]:
def fit_bertopic(df, column_index):
    """
    Fit BERTopic on one dataframe column.

    Returns:
        topic_model: fitted BERTopic model
        topics: topic assignment list for each document
        probs: topic probability matrix from BERTopic
        docs: list of documents used for fitting
    """
    # Pull selected column, keeping only non-empty responses
    prompt_label = str(df.columns[column_index])
    responses = df.iloc[:, column_index]
    valid_mask = responses.notna() & responses.astype(str).str.strip().ne("")
    docs = responses[valid_mask].astype(str).tolist()

    if len(docs) == 0:
        raise ValueError(f"No non-empty responses found in column index {column_index}.")

    print(f"Running BERTopic for prompt: {prompt_label}")

    # UMAP setup
    umap_model = UMAP(
        n_neighbors=10,
        n_components=5,
        min_dist=0.0,
        metric="cosine",
        random_state=42
    )

    # HDBSCAN setup
    hdb = hdbscan.HDBSCAN(
        min_cluster_size=5,
        min_samples=2,
        prediction_data=True,
        cluster_selection_epsilon=0.1
    )

    # OpenAI/Ollama-backed representation model setup
    client = openai.OpenAI(
        base_url="http://localhost:11434/v1",
        api_key="ollama"
    )
    representation_model = OpenAI(client, model="mistral")

    # Fit BERTopic
    topic_model = BERTopic(
        embedding_model="all-MiniLM-L6-v2",
        umap_model=umap_model,
        n_gram_range=(1, 5),
        calculate_probabilities=True,
        hdbscan_model=hdb,
        verbose=True,
        representation_model=representation_model
    )
    topics, probs = topic_model.fit_transform(docs)

    return topic_model, topics, probs, docs

In [38]:
# 
q1_topic_model, q1_topics, q1_probs, q1_docs = fit_bertopic(
    df=df,
    column_index=13
)

2026-04-06 12:30:40,689 - BERTopic - Embedding - Transforming documents to embeddings.


Running BERTopic for prompt: In this class you ll be learning about the process of scientific investigation What do you think that process looks like from the perspective of a researcher compared to what it looks like from the perspective of someone in the general public who is a consumer of scientific results Write a paragraph at least 3 5 sentences about how you think science happens


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4737.61it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 20/20 [00:06<00:00,  3.02it/s]
2026-04-06 12:30:55,643 - BERTopic - Embedding - Completed ✓
2026-04-06 12:30:55,643 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
/Users/tylerwiederich/Library/CloudStorage/OneDrive-UniversityofNebraska-Lincoln/4 - Obsidian Vault/Research/dissertation/ch2-experiential-learning/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
2026-04-06 12:30:56,528 - BERTopic - Dimensionality - Completed ✓
2026-04-06 12:30:56,

# Paper Draft

## Methods

**NOTE:** This section is only for these models. I have more in the actual manuscript.

Our sample is composed of students enrolled in STAT 218 at the University of Nebraska--Lincoln. While all students were required to participant in the experiential learning project as part of the course cirriculum, data was collected if students meet the age of majority in Nebraska (age 19 or older) and if they consented to data collection. The data collection took place between Summer 2023 and Spring 2025, where XX sections of STAT 218 participated. 


### Text Classification

In recent years, there has been a growing research area in Large-Language Models (LLMs) for classifying open-ended survey responses. Despite the promising aspect of using LLMs for classification, these models are sensitive to prompt and token limits (XXX), which can greatly influence the outputs. Additionally, these models tend to suffer from hallucinations and/or low accuracy rates compared to traditional human codings (XXX). The current capacity of LLMs are not yet ready for standalone classification, but they do have some integrations with non-zero box methods. 

For the classiciation of our responses, we focus on BERTopic (XXX), which is a topic modeling algorithm that incorporates clustering and dimensional reduction. BERTopic is highly customizable in each stage of the algorithm, allowing for multiple specifications for fine-tuning. Descriptions of this process can be found in Table (XXX), along with our specified models and hyper-parameters for each stage. We note a few of our chosen hyperparameter selections. For the clustering stage with HBDSCAN, we set the minimum cluster size to 5 so that smaller clusters can form. We found that the default settings were too restrictive and formed topics that closely aligned with the original prompt. We also included n-grams up to five words to allow for common phrases that students may have responded with (e.g., "scientific process"). Lastly, we incorporated Mistral (XXX) as a locally-run LLM to fine-tune the generated topic lists into interpretable categories, while also respecting concerns over data privacy of cloud-based LLMs.




| Stage | Pipeline Step | Purpose | Option Used |
|---|---|---|---|
| Stage 1 | extract embeddings | Convert each response into a numeric vector that captures semantic meaning. | `embedding_model="all-MiniLM-L6-v2"` |
| Stage 2 | reduce dimensionality | Compress embeddings into a lower-dimensional space for better clustering efficiency | `UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric="cosine", random_state=42)` |
| Stage 3 | cluster reduced embeddings | Group similar responses into topic clusters. | `HDBSCAN(min_cluster_size=5, min_samples=2, cluster_selection_epsilon=0.1, prediction_data=True)` |
| Stage 4 | tokenize topics | Break text into candidate terms/phrases used to represent each cluster. | `n_gram_range=(1, 5)` |
| Stage 5 | extract topic words | Compute the most representative words/phrases for each cluster. | BERTopic default c-TF-IDF weighting |
| Stage 6 | fine-tune topic representations | Improve readability and specificity of topic labels/keywords. | `representation_model=OpenAI(client, model="mistral")` with Ollama endpoint `http://localhost:11434/v1` |

## Results

### Pre-Experiment

The pre-experiment prompt consisted of one question asking participants 

> In this class, you’ll be learning about the process of scientific investigation. What do you think that process looks like, from the perspective of a researcher, compared to what it looks like from the perspective of someone in the general public who is a consumer of scientific results? Write a paragraph (at least 3-5 sentences) about how you think science happens.


In [ ]:
# Q1
q1_topic_model, q1_topics, q1_probs, q1_docs = fit_bertopic(
    df=df,
    column_index=13
)



100%|██████████| 17/17 [01:09<00:00,  4.06s/it]
2026-04-06 12:12:45,737 - BERTopic - Representation - Completed ✓


In [ ]:
q1_topic_model.visualize_topics()

/Users/tylerwiederich/Library/CloudStorage/OneDrive-UniversityofNebraska-Lincoln/4 - Obsidian Vault/Research/dissertation/ch2-experiential-learning/.venv/lib/python3.11/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [44]:
hierarchical_topics = topic_model.hierarchical_topics(q1)
topic_model.visualize_hierarchy(hierarchical_topics=hierarchical_topics)

100%|██████████| 7/7 [00:10<00:00,  1.56s/it]


### Post-Experiment

### Abstract Reflection

### Presentation Reflection